# Structured RAG + Hypothetical Document Embeddings LAB

*To use this notebook, go to Kernel -> Change Kernel. Select switch to **Agentic Search Course Environment** *

This notebook demonstrates structured RAG as an approach

* We use pydantic to get a structured output from the search engine
* That becomes our query
* We can use that to create "Hypothetical Document" which can be used to generate an embedding
* This embedding *looks like* other documents, so its possible it will work better when searching other document embeddings.

In [1]:
import os
os.environ["CHEAT_AT_SEARCH_DATA_PATH"] = "/home/jovyan/data"

from cheat_at_search.wands_data import corpus, judgments

In [2]:
corpus

,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count,features,doc_id,title,description,category,sub_category,cat_subcat
0,0,solid wood platform bed,Beds,Furniture / Bedroom Furniture / Beds & Headboa...,"good , deep sleep can be quite difficult to ha...",overallwidth-sidetoside:64.7|dsprimaryproducts...,15.0,4.5,15.0,"[overallwidth-sidetoside:64.7, dsprimaryproduc...",0,solid wood platform bed,"good , deep sleep can be quite difficult to ha...",Furniture,Bedroom Furniture,Furniture / Bedroom Furniture
1,1,all-clad 7 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,"create delicious slow-cooked meals , from tend...",capacityquarts:7|producttype : slow cooker|pro...,100.0,2.0,98.0,"[capacityquarts:7, producttype : slow cooker, ...",1,all-clad 7 qt . slow cooker,"create delicious slow-cooked meals , from tend...",Kitchen & Tabletop,Small Kitchen Appliances,Kitchen & Tabletop / Small Kitchen Appliances
2,2,all-clad electrics 6.5 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,prepare home-cooked meals on any schedule with...,features : keep warm setting|capacityquarts:6....,208.0,3.0,181.0,"[features : keep warm setting, capacityquarts:...",2,all-clad electrics 6.5 qt . slow cooker,prepare home-cooked meals on any schedule with...,Kitchen & Tabletop,Small Kitchen Appliances,Kitchen & Tabletop / Small Kitchen Appliances
3,3,all-clad all professional tools pizza cutter,"Slicers, Peelers And Graters",Browse By Brand / All-Clad,this original stainless tool was designed to c...,overallwidth-sidetoside:3.5|warrantylength : l...,69.0,4.5,42.0,"[overallwidth-sidetoside:3.5, warrantylength :...",3,all-clad all professional tools pizza cutter,this original stainless tool was designed to c...,Browse By Brand,All-Clad,Browse By Brand / All-Clad
4,4,baldwin prestige alcott passage knob with roun...,Door Knobs,Home Improvement / Doors & Door Hardware / Doo...,the hardware has a rich heritage of delivering...,compatibledoorthickness:1.375 '' |countryofori...,70.0,5.0,42.0,"[compatibledoorthickness:1.375 '' , countryofo...",4,baldwin prestige alcott passage knob with roun...,the hardware has a rich heritage of delivering...,Home Improvement,Doors & Door Hardware,Home Improvement / Doors & Door Hardware
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42989,42989,malibu pressure balanced diverter fixed shower...,Shower Panels,Home Improvement / Bathroom Remodel & Bathroom...,the malibu pressure balanced diverter fixed sh...,producttype : shower panel|spraypattern : rain...,3.0,4.5,2.0,"[producttype : shower panel, spraypattern : ra...",42989,malibu pressure balanced diverter fixed shower...,the malibu pressure balanced diverter fixed sh...,Home Improvement,Bathroom Remodel & Bathroom Fixtures,Home Improvement / Bathroom Remodel & Bathroom...
42990,42990,emmeline 5 piece breakfast dining set,Dining Table Sets,Furniture / Kitchen & Dining Furniture / Dinin...,,basematerialdetails : steel| : gray wood|ofhar...,1314.0,4.5,864.0,"[basematerialdetails : steel, : gray wood, of...",42990,emmeline 5 piece breakfast dining set,,Furniture,Kitchen & Dining Furniture,Furniture / Kitchen & Dining Furniture
42991,42991,maloney 3 piece pub table set,Dining Table Sets,Furniture / Kitchen & Dining Furniture / Dinin...,this pub table set includes 1 counter height t...,additionaltoolsrequirednotincluded : power dri...,49.0,4.0,41.0,[additionaltoolsrequirednotincluded : power dr...,42991,maloney 3 piece pub table set,this pub table set includes 1 counter height t...,Furniture,Kitchen & Dining Furniture,Furniture / Kitchen & Dining Furniture
42992,42992,fletcher 27.5 '' wide polyester armchair,Teen Lounge Furniture|Accent Chairs,Furniture / Living Room Furniture / Chairs & S...,"bring iconic , modern style to your space in a...",legmaterialdetails : rubberwood|backheight-sea...,1746.0,4.5,1226.0,"[legmateri

## Get OpenAI keys

Load OpenAI Keys for interacting with results.

In [3]:
from openai import OpenAI
from cheat_at_search.data_dir import key_for_provider

openai_key = key_for_provider("openai")
openai = OpenAI(api_key=openai_key)

You're going to be prompted for your API key. This will be stored in a local file
If you'd prefer to set it as an environment variable, set it as:
    export OPENAI_API_KEY=your_api_key_here


Enter your openai_api_key:  ········


## Structured RAG

We'll use a structured Pydantic request to generate queries.

In [4]:
from pydantic import BaseModel, Field
from typing import Literal

Category = Literal['Furniture',
 'Home Improvement',
 'Décor & Pillows',
 'Outdoor',
 'Storage & Organization',
 'Lighting',
 'Rugs',
 'Bed & Bath',
 'Kitchen & Tabletop',
 'Baby & Kids',
 'School Furniture and Supplies',
 'Appliances',
 'Holiday Décor',
 'Commercial Business Furniture',
 'Pet',
 'Contractor']

class WayfairProductQuery(BaseModel):
    """A query for the wayfair product dataset."""

    search_query: str = Field(..., description="a keyword search for what the user wants")

    category: Category = Field(..., description="the category of the product")

## Generate query using the structure

In [5]:
from openai import OpenAI
from cheat_at_search.data_dir import key_for_provider
import pandas as pd

model = "gpt-5.4-mini"

query = "I would like a geometric sofa, ideally red leather, with interesting stitching"

inputs = [
    {"role": "system", "content": "You are a helpful assistant that generates e-commerce searches for user requests."},
    {"role": "user", "content": "Generate search query for the following query: " + query},
]

resp = openai.responses.parse(
    model=model,
    input=inputs,
    text_format=WayfairProductQuery
)


resp.output[-1].content[-1].parsed

WayfairProductQuery(search_query='geometric sofa red leather interesting stitching', category='Furniture')

## Hypothetical Document Embeddings (HyDE)

Ask OpenAI for a fake document we then use to embed in document space alongside the other existing document embeddings.

Below we

* Give some examples of real products
* Use structured outputs to ask OpenAI to give us product metadata
* We embed the generated document
* We see whats the most similar

In [6]:
from openai import OpenAI
from cheat_at_search.data_dir import key_for_provider
import pandas as pd

model = "gpt-5.4-mini"

query = "I would like a geometric sofa, ideally red leather, with interesting stitching"

real_products = """
Title: esmarie 12 '' console table
Description:

Title: umberger embroidery comforter set
Description: the beautifully tufted bed . its gray coloring makes this set easy to accessorize in your bedroom .

Title: jarod tv stand for tvs up to 78 ''
Description: make the focal point of your living room a stylish and organized one with this tv stand . the solid colors of the tv stand , coupled with the wood grain look of the doors and simple silhouette , are hallmarks of its mid-century modern-inspired design . the stand 's engineered wood frame is built to last . this collection features six convenient shelf spaces behind three light oak finished doors . the back panel of the television stand has circular cut-outs for wire management , should you use the shelves to store gaming consoles or media players . the knobs of this collection are delightful square knobs finished in a dark brown . the brown-finished legs of the tv stand are angled and tapered , further exemplifying its mid-century modern vibes . this collection makes it easy to keep your living room or rec room organized and tidy with its sleek look and storage options . made in malaysia , the tv stand requires assembly .

Title: mccampbell abstract navy blue/white/gray area rug
Description: artistically hand carved 3d effect , thick and high pile rug designed with unique colors that bring out the beauty and luxury of one of the best selling area rugs in the usa , plush , soft and durable , easy to maintain .

"""


class WayfairProduct(BaseModel):
    """A product in the wayfair home goods / furniture dataset."""

    title: str = Field(..., description="product name")
    description: str = Field(..., description="product description")


inputs = [
    {"role": "system",
     "content": "You are a helpful assistant that generates hypothetical furniture / home goods products that might solve a user's query based on examples."},
    {"role": "user",
     "content": "Some hypothetical products " + real_products},
    {"role": "user",
     "content": "Generate a hypothetical document based on these queries: " + query},
]

resp = openai.responses.parse(
    model=model,
    input=inputs,
    text_format=WayfairProduct
)


hypothetical_product = resp.output[-1].content[-1].parsed
hypothetical_product

WayfairProduct(title='kessler geometric red leather sofa', description='bring a bold contemporary look to your living room with this geometric sofa in rich red faux leather . the streamlined silhouette features angular lines and a sculptural profile that gives the piece a modern architectural feel . detailed contrast stitching adds visual interest across the seat cushions and arms , while the supportive foam-filled cushions offer comfortable everyday seating . sleek tapered legs complete the design , making this sofa an eye-catching centerpiece for any stylish space .')

### Actually search

Now we actually search with the embeddings.

Do we like the results?

In [7]:
from sentence_transformers import SentenceTransformer, util
from cheat_at_search.wands_data import product_embeddings

minilm = SentenceTransformer('all-MiniLM-L6-v2')
product_text = f"""## {hypothetical_product.title}

 {hypothetical_product.description}"""
product_embedding = minilm.encode(product_text)

/opt/conda/envs/course-environment/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import numpy

dotted = numpy.dot(product_embedding, product_embeddings.T)
top_k = numpy.argsort(dotted)[-10:]

corpus.iloc[top_k]

,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count,features,doc_id,title,description,category,sub_category,cat_subcat
10806,10806,maynard 80.7 '' faux leather square arm sofa,Reception Sofas & Loveseats|Sofas,Furniture / Living Room Furniture / Sofas,"versatile and durable , the product can be use...",armheight-floortoarm:25.2|upholsterycolor : bl...,16.0,5.0,14.0,"[armheight-floortoarm:25.2, upholsterycolor : ...",10806,maynard 80.7 '' faux leather square arm sofa,"versatile and durable , the product can be use...",Furniture,Living Room Furniture,Furniture / Living Room Furniture
38685,38685,chafin contemporary leather sofa,Reception Sofas & Loveseats|Sofas,Reception Area / Reception Seating / Reception...,your office reception space speaks to who your...,armmaterial : steel|backwidth-sidetoside:67.5|...,154.0,4.5,99.0,"[armmaterial : steel, backwidth-sidetoside:67....",38685,chafin contemporary leather sofa,your office reception space speaks to who your...,Reception Area,Reception Seating,Reception Area / Reception Seating
41298,41298,umer 66.9 '' faux leather tuxedo arm loveseat,Sofas,Furniture / Living Room Furniture / Sofas,"this 3-seater sofa , made from quality artific...",backheight-seattotopofback:8.66|levelofassembl...,2.0,3.0,2.0,"[backheight-seattotopofback:8.66, levelofassem...",41298,umer 66.9 '' faux leather tuxedo arm loveseat,"this 3-seater sofa , made from quality artific...",Furniture,Living Room Furniture,Furniture / Living Room Furniture
25036,25036,quarles 30 '' wide faux leather manual glider ...,Recliners,Furniture / Living Room Furniture / Chairs & S...,this product instantly makes any room in your ...,ottomanwidth-sidetoside:15.5|minimumdoorwidth-...,533.0,4.5,425.0,"[ottomanwidth-sidetoside:15.5, minimumdoorwidt...",25036,quarles 30 '' wide faux leather manual glider ...,this product instantly makes any room in your ...,Furniture,Living Room Furniture,Furniture / Living Room Furniture
17552,17552,faux leather modular seating component sectional,Sectionals,Furniture / Living Room Furniture / Sectionals...,they let you mix and match components and fabr...,armheight-floortoarm:28.5|onearmedchairheight:...,NaN,NaN,NaN,"[armheight-floortoarm:28.5, onearmedchairheigh...",17552,faux leather modular seating component sectional,they let you mix and match components and fabr...,Furniture,Living Room Furniture,Furniture / Living Room Furniture
20568,20568,spuyten 109 '' wide faux leather right hand fa...,Sofas|Sectionals,Furniture / Living Room Furniture / Sectionals,anchor your favorite seating space in transiti...,dssecondaryproductstyle : ultra-modern|seatdep...,49.0,4.5,32.0,"[dssecondaryproductstyle : ultra-modern, seatd...",20568,spuyten 109 '' wide faux leather right hand fa...,anchor your favorite seating space in transiti...,Furniture,Living Room Furniture,Furniture / Living Room Furniture
1743,1743,russ 103.5 '' wide faux leather sofa & chaise ...,Sectionals,Furniture / Living Room Furniture / Sectionals,from entertaining guests to helping us host th...,backtype : tufted back|legmaterial : plastic|o...,4010.0,4.5,2544.0,"[backtype : tufted back, legmaterial : plastic...",1743,russ 103.5 '' wide faux leather sofa & chaise ...,from entertaining guests to helping us host th...,Furniture,Living Room Furniture,Furniture / Living Room Furniture
5129,5129,85 '' faux leather tuxedo arm sofa,Reception Sofas & Loveseats|Sofas,Furniture / Living Room Furniture / Sofas,rejuvenate your living space with the addition...,design : standard|upholsterymaterial : faux le...,NaN,NaN,NaN,"[design : standard, upholsterymaterial : faux ...",5129,85 '' faux leather tuxedo arm sofa,rejuvenate your living space with the addition...,Furniture,Living Room Furniture,Furniture / Living Room Furniture
30152,30152,krehbiel 72 '' faux leather flared arm sofa,Reception Sofas & Loveseats|Sofas,Furniture / Living Room Furniture / Sofas,stunningly sharp 

## How could this be improved

* Should we only do title embeddings?
* Should we use a lexical approach (ie [more like this](https://docs.opensearch.org/latest/query-dsl/specialized/more-like-this/)) instead?